In [1]:
"""
Baseline ResNet-50 for Thoracic Disease Classification
Based on: El-Fiky et al. (2021) - Multi-Label Transfer Learning

This notebook implements the baseline ResNet-50 model using transfer learning
methodology from the paper to achieve comparable performance (AUC: 0.911, F1: 0.66)
"""
import numpy as np
import pandas as pd
import os
import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, hamming_loss, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.10.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================================
# CONFIGURATION - Following El-Fiky et al. (2021) Paper
# ============================================================================

# Dataset configuration
IMAGE_SIZE = 224  # Paper uses 224x224
BATCH_SIZE = 32   # Paper uses batch size 32
EPOCHS = 60       # Paper uses 60 epochs
LEARNING_RATE = 0.001  # Paper uses 0.001

# Disease labels (14 diseases + No Finding)
CLASS_NAMES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
    'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
NUM_CLASSES = len(CLASS_NAMES)

print(f"\nConfiguration:")
print(f"  Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Number of Classes: {NUM_CLASSES}")


Configuration:
  Image Size: 224x224
  Batch Size: 32
  Epochs: 60
  Learning Rate: 0.001
  Number of Classes: 14


In [3]:
# ============================================================================
# DATA LOADING
# ============================================================================

BASE_DIR = "./dataset_balanced/"
CSV_PATH = os.path.join(BASE_DIR, "new_labels.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "new_images", "new_images")

print(f"\nLoading dataset from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Ensure all class columns exist
for class_name in CLASS_NAMES:
    if class_name not in df.columns:
        print(f"Warning: {class_name} not found in dataset, adding as zeros")
        df[class_name] = 0

# Create full image paths
def create_image_path(filename):
    return os.path.join(IMAGE_DIR, filename)

if 'Image Index' in df.columns:
    df['image_path'] = df['Image Index'].apply(create_image_path)
elif 'Path' in df.columns:
    df['image_path'] = df['Path'].apply(create_image_path)
else:
    df['image_path'] = df.iloc[:, 0].apply(create_image_path)

# Verify images exist
valid_indices = []
missing_count = 0

for idx, row in df.iterrows():
    if os.path.exists(row['image_path']):
        valid_indices.append(idx)
    else:
        missing_count += 1

df = df.iloc[valid_indices].reset_index(drop=True)
print(f"Valid images: {len(df)}")
print(f"Missing images: {missing_count}")

# Display class distribution
print("\nClass Distribution:")
print("="*60)
for class_name in CLASS_NAMES:
    positive_count = df[class_name].sum()
    percentage = (positive_count / len(df)) * 100
    print(f"{class_name:20s}: {positive_count:6d} ({percentage:5.2f}%)")

# Calculate class weights for imbalanced data (Paper addresses this)
class_weights = {}
total_samples = len(df)
for i, class_name in enumerate(CLASS_NAMES):
    positive_count = df[class_name].sum()
    if positive_count > 0:
        # Inverse frequency weighting
        class_weights[i] = total_samples / (NUM_CLASSES * positive_count)
    else:
        class_weights[i] = 1.0

print("\nClass Weights (for imbalance handling):")
for i, class_name in enumerate(CLASS_NAMES):
    print(f"{class_name:20s}: {class_weights[i]:.3f}")


Loading dataset from: ./dataset_balanced/new_labels.csv
Valid images: 51382
Missing images: 0

Class Distribution:
Atelectasis         :   4788 ( 9.32%)
Cardiomegaly        :   4629 ( 9.01%)
Consolidation       :   4481 ( 8.72%)
Edema               :   4554 ( 8.86%)
Effusion            :   5000 ( 9.73%)
Emphysema           :   4467 ( 8.69%)
Fibrosis            :   5039 ( 9.81%)
Hernia              :   5590 (10.88%)
Infiltration        :   5740 (11.17%)
Mass                :   5564 (10.83%)
Nodule              :   5634 (10.96%)
Pleural_Thickening  :   4831 ( 9.40%)
Pneumonia           :   6105 (11.88%)
Pneumothorax        :   4919 ( 9.57%)

Class Weights (for imbalance handling):
Atelectasis         : 0.767
Cardiomegaly        : 0.793
Consolidation       : 0.819
Edema               : 0.806
Effusion            : 0.734
Emphysema           : 0.822
Fibrosis            : 0.728
Hernia              : 0.657
Infiltration        : 0.639
Mass                : 0.660
Nodule              : 0.651
Ple

In [4]:
# ============================================================================
# TRAIN/TEST SPLIT - Paper uses 80/20 split
# ============================================================================

print("\nSplitting dataset (80% train, 20% test)...")
train_df, val_df = train_test_split(
    df, 
    test_size=0.2,  # 80/20 split as per papers
    random_state=42,
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


Splitting dataset (80% train, 20% test)...
Training samples: 41105
Validation samples: 10277


In [5]:
def create_tf_dataset(df, batch_size, shuffle=True, augment=True):
    """Enhanced augmentation matching paper methodology"""
    
    def load_and_preprocess(image_path, labels):
        img = tf.io.read_file(image_path)
        img = tf.image.decode_png(img, channels=3)  # Changed to PNG
        img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
        
        if augment:
            # More aggressive augmentation (papers typically use this)
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.2)
            img = tf.image.random_contrast(img, 0.8, 1.2)
            
            # Random rotation (small angles to preserve anatomy)
            angle = tf.random.uniform([], -0.1, 0.1)  # ~±6 degrees
            img = tf.image.rotate(img, angle) if 'tfa' in dir() else img
        
        img = tf.keras.applications.resnet50.preprocess_input(img)
        return img, labels
    
    image_paths = df['image_path'].values
    labels = df[CLASS_NAMES].values.astype(np.float32)
    
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=42)
    
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Replace your generators with this:
train_dataset = create_tf_dataset(train_df, BATCH_SIZE, shuffle=True, augment=True)
val_dataset = create_tf_dataset(val_df, BATCH_SIZE, shuffle=False, augment=False)

print(f"Optimized tf.data pipelines created")

Optimized tf.data pipelines created


In [6]:
# ============================================================================
# MODEL ARCHITECTURE - ResNet-50 Transfer Learning (Paper Methodology)
# ============================================================================

def create_baseline_resnet50(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    num_classes=NUM_CLASSES,
    dropout_rate=0.3  # Reduced from 0.5
):
    """
    Paper methodology:
    - ResNet50 pre-trained on ImageNet
    - Global Average Pooling
    - Dense layer (paper doesn't specify size, using 512)
    - Dropout for regularization
    - Output layer with sigmoid (multi-label)
    """
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'  # Global Average Pooling
    )
    
    # Initially freeze (we'll unfreeze in stage 2)
    base_model.trainable = False
    
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    
    # Classification head (simpler than your current version)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(512, activation='relu')(x)  # Removed L2 regularization
    x = layers.Dropout(dropout_rate)(x)
    
    outputs = layers.Dense(num_classes, activation='sigmoid', dtype='float32')(x)
    
    model = keras.Model(inputs, outputs, name='Baseline_ResNet50')
    return model

print("\nCreating Baseline ResNet-50 model...")
model = create_baseline_resnet50()

# Model summary
print(f"\nModel Architecture:")
print(f"  Total parameters: {model.count_params():,}")
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"  Trainable parameters: {trainable_params:,}")


Creating Baseline ResNet-50 model...

Model Architecture:
  Total parameters: 24,643,982
  Trainable parameters: 1,056,270


In [ ]:
# ============================================================================
# TWO-STAGE TRAINING STRATEGY (Following Transfer Learning Best Practices)
# ============================================================================

print("\n" + "="*80)
print("STAGE 1: Train Classification Head (Frozen ResNet50)")
print("="*80)

# Stage 1: Train only the classifier head
model.compile(
    optimizer=Adam(learning_rate=1e-3),  # Higher LR for new layers
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

stage1_callbacks = [
    ModelCheckpoint(
        filepath='./output/baseline_stage1_best.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_auc',
        patience=5,
        mode='max',
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

history_stage1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,  # Warm-up phase
    callbacks=stage1_callbacks,
    verbose=1
)

print("\n" + "="*80)
print("STAGE 2: Fine-tune Entire Network (Unfrozen ResNet50)")
print("="*80)

# Stage 2: Unfreeze and fine-tune entire network
base_model = model.layers[1]  # Get the ResNet50 base
base_model.trainable = True

# Optional: Freeze batch normalization layers to prevent instability
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(f"Unfrozen ResNet50 - Now training {sum([1 for l in base_model.layers if l.trainable])} layers")

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Much lower LR for fine-tuning
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

stage2_callbacks = [
    ModelCheckpoint(
        filepath='./output/baseline_resnet50_best.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_auc',
        patience=15,  # More patience for fine-tuning
        mode='max',
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    )
]

history_stage2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,  # Fine-tuning phase
    initial_epoch=10,  # Continue from stage 1
    callbacks=stage2_callbacks,
    verbose=1
)

# Combine histories
history = history_stage1
for key in history_stage2.history.keys():
    history.history[key].extend(history_stage2.history[key])

# Save final model
model.save('./output/baseline_resnet50_final.keras')
print("\nFinal model saved!")


STAGE 1: Train Classification Head (Frozen ResNet50)
Epoch 1/10
1285/1285 [==============================] - ETA: 0s - loss: 0.2981 - binary_accuracy: 0.9029 - auc: 0.6877 - precision: 0.5742 - recall: 0.0779
Epoch 1: val_auc improved from -inf to 0.75310, saving model to ./output\baseline_stage1_best.keras
1285/1285 [==============================] - 173s 129ms/step - loss: 0.2981 - binary_accuracy: 0.9029 - auc: 0.6877 - precision: 0.5742 - recall: 0.0779 - val_loss: 0.2745 - val_binary_accuracy: 0.9062 - val_auc: 0.7531 - val_precision: 0.7659 - val_recall: 0.0842 - lr: 0.0010
Epoch 2/10
1284/1285 [============================>.] - ETA: 0s - loss: 0.2821 - binary_accuracy: 0.9066 - auc: 0.7232 - precision: 0.6925 - recall: 0.1030
Epoch 2: val_auc improved from 0.75310 to 0.76878, saving model to ./output\baseline_stage1_best.keras
1285/1285 [==============================] - 192s 149ms/step - loss: 0.2821 - binary_accuracy: 0.9066 - auc: 0.7232 - precision: 0.6925 - recall: 0.1030 

In [ ]:
# ============================================================================
# TRAINING HISTORY VISUALIZATION
# ============================================================================

def plot_training_history(history):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Training Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # AUC
    axes[0, 1].plot(history.history['auc'], label='Training AUC')
    axes[0, 1].plot(history.history['val_auc'], label='Validation AUC')
    axes[0, 1].set_title('Model AUC')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AUC')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Training Precision')
    axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
    axes[1, 0].set_title('Model Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Training Recall')
    axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
    axes[1, 1].set_title('Model Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./output/baseline_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION - Matching Paper Metrics
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION")
print("="*80)

# Generate predictions on validation dataset
print("Generating predictions on validation set...")
predictions = model.predict(val_dataset, verbose=1)

# Get true labels from validation dataframe
y_true = val_df[CLASS_NAMES].values

print(f"\nPredictions shape: {predictions.shape}")
print(f"True labels shape: {y_true.shape}")

In [ ]:
# ============================================================================
# PER-CLASS AUC SCORES (Table 3 in Paper)
# ============================================================================

print("\n" + "="*70)
print("TABLE 3: PER-CLASS AUC SCORES (Comparison with El-Fiky et al. 2021)")
print("="*70)
print(f"{'Disease':<25s} {'Our AUC':>12s} {'Paper AUC':>12s} {'Difference':>12s}")
print("-"*70)

# Paper's reported AUC values from El-Fiky et al. (2021) Table 3
paper_aucs = {
    'Atelectasis': 0.896,
    'Cardiomegaly': 0.934,
    'Consolidation': 0.871,
    'Edema': 0.943,
    'Effusion': 0.936,
    'Emphysema': 0.941,
    'Fibrosis': 0.931,
    'Hernia': 0.873,
    'Infiltration': 0.845,
    'Mass': 0.932,
    'Nodule': 0.894,
    'Pleural_Thickening': 0.911,
    'Pneumonia': 0.915,
    'Pneumothorax': 0.934
}

per_class_aucs = []
for i, class_name in enumerate(CLASS_NAMES):
    try:
        # Check if class has both positive and negative samples
        if len(np.unique(y_true[:, i])) > 1:
            auc_score = roc_auc_score(y_true[:, i], predictions[:, i])
            per_class_aucs.append(auc_score)
            
            paper_auc = paper_aucs.get(class_name, 0.0)
            diff = auc_score - paper_auc
            
            print(f"{class_name:<25s} {auc_score:>12.4f} {paper_auc:>12.4f} {diff:>+12.4f}")
        else:
            per_class_aucs.append(0.0)
            print(f"{class_name:<25s} {'N/A':>12s} {'N/A':>12s} {'N/A':>12s}")
    except Exception as e:
        per_class_aucs.append(0.0)
        print(f"{class_name:<25s} {'ERROR':>12s} {'ERROR':>12s} {'ERROR':>12s}")
        print(f"  Error details: {str(e)}")

# Calculate average AUC
valid_aucs = [auc for auc in per_class_aucs if auc > 0]
avg_auc = np.mean(valid_aucs) if valid_aucs else 0.0
paper_avg_auc = 0.911

print("-"*70)
print(f"{'AVERAGE AUC':<25s} {avg_auc:>12.4f} {paper_avg_auc:>12.4f} {avg_auc-paper_avg_auc:>+12.4f}")
print("="*70)

# Performance assessment
print("\nPerformance Assessment:")
if avg_auc >= 0.90:
    print("✓ EXCELLENT: Comparable to paper performance!")
    status = "success"
elif avg_auc >= 0.85:
    print("~ GOOD: Close to paper performance")
    status = "acceptable"
else:
    print("⚠ NEEDS IMPROVEMENT: Below paper performance")
    status = "needs_work"
    
print(f"  Your AUC: {avg_auc:.4f}")
print(f"  Paper AUC: {paper_avg_auc:.4f}")
print(f"  Gap: {abs(avg_auc - paper_avg_auc):.4f} ({((avg_auc - paper_avg_auc)/paper_avg_auc)*100:+.2f}%)")

In [ ]:
# ============================================================================
# TABLE 2: OVERALL METRICS (Micro F1-score and Hamming Loss)
# ============================================================================

# Binary predictions at 0.5 threshold (as per paper)
y_pred_binary = (predictions > 0.5).astype(int)

# Calculate metrics following paper methodology
from sklearn.metrics import f1_score, precision_score, recall_score

micro_f1 = f1_score(y_true, y_pred_binary, average='micro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred_binary, average='macro', zero_division=0)
micro_precision = precision_score(y_true, y_pred_binary, average='micro', zero_division=0)
micro_recall = recall_score(y_true, y_pred_binary, average='micro', zero_division=0)
hamming = hamming_loss(y_true, y_pred_binary)

print("\n" + "="*70)
print("TABLE 2: OVERALL PERFORMANCE METRICS (El-Fiky et al. 2021)")
print("="*70)
print(f"{'Metric':<25s} {'Our Score':>15s} {'Paper Score':>15s} {'Difference':>12s}")
print("-"*70)
print(f"{'Average AUC':<25s} {avg_auc:>15.4f} {0.911:>15.4f} {avg_auc-0.911:>+12.4f}")
print(f"{'Micro F1-score':<25s} {micro_f1:>15.4f} {0.660:>15.4f} {micro_f1-0.660:>+12.4f}")
print(f"{'Macro F1-score':<25s} {macro_f1:>15.4f} {'N/A':>15s} {'N/A':>12s}")
print(f"{'Micro Precision':<25s} {micro_precision:>15.4f} {'N/A':>15s} {'N/A':>12s}")
print(f"{'Micro Recall':<25s} {micro_recall:>15.4f} {'N/A':>15s} {'N/A':>12s}")
print(f"{'Hamming Loss':<25s} {hamming:>15.4f} {0.404:>15.4f} {hamming-0.404:>+12.4f}")
print("="*70)

# Performance summary
print("\nPerformance Summary:")
meets_auc = "✓" if avg_auc >= 0.90 else "✗"
meets_f1 = "✓" if micro_f1 >= 0.60 else "✗"
meets_hamming = "✓" if hamming <= 0.45 else "✗"

print(f"  {meets_auc} Average AUC:    {avg_auc:.4f} (Target: ≥0.90)")
print(f"  {meets_f1} Micro F1:       {micro_f1:.4f} (Target: ≥0.60)")
print(f"  {meets_hamming} Hamming Loss:  {hamming:.4f} (Target: ≤0.45)")

In [ ]:
# ============================================================================
# DETAILED CLASSIFICATION REPORT
# ============================================================================

print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_true, 
    y_pred_binary, 
    target_names=CLASS_NAMES,
    zero_division=0,
    digits=4
))

In [ ]:
# ============================================================================
# ROC CURVES (Figure 4 in Paper)
# ============================================================================

def plot_roc_curves_combined(y_true, predictions, class_names):
    """Plot all ROC curves on a single graph with different colors"""
    
    # Create figure
    plt.figure(figsize=(12, 10))
    
    # Color palette - use a colormap for distinct colors
    colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
    
    # Plot ROC curve for each class
    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        try:
            if len(np.unique(y_true[:, i])) > 1:
                fpr, tpr, _ = roc_curve(y_true[:, i], predictions[:, i])
                auc_score = roc_auc_score(y_true[:, i], predictions[:, i])
                
                plt.plot(fpr, tpr, linewidth=2.5, color=color, 
                        label=f'{class_name} (AUC = {auc_score:.3f})', alpha=0.8)
        except Exception as e:
            print(f"Error plotting {class_name}: {str(e)}")
    
    # Plot diagonal (random classifier)
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier', alpha=0.5)
    
    # Formatting
    plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    plt.title('ROC Curves - All 14 Thoracic Diseases\n(El-Fiky et al. 2021 Baseline)', 
             fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right', fontsize=10, framealpha=0.9)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    
    # Add average AUC text
    valid_aucs = [roc_auc_score(y_true[:, i], predictions[:, i]) 
                  for i in range(len(class_names)) 
                  if len(np.unique(y_true[:, i])) > 1]
    avg_auc_plot = np.mean(valid_aucs) if valid_aucs else 0.0
    
    plt.text(0.4, 0.2, f'Average AUC: {avg_auc_plot:.4f}\nPaper AUC: 0.9110', 
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            verticalalignment='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('./output/baseline_roc_curves_combined.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Combined ROC curve saved to: ./output/baseline_roc_curves_combined.png")


def plot_roc_curves_grid(y_true, predictions, class_names):
    """Plot ROC curves in a 4x4 grid (original individual plots)"""
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.ravel()
    
    for i, class_name in enumerate(class_names):
        if i < len(axes):
            try:
                if len(np.unique(y_true[:, i])) > 1:
                    fpr, tpr, _ = roc_curve(y_true[:, i], predictions[:, i])
                    auc_score = roc_auc_score(y_true[:, i], predictions[:, i])
                    
                    axes[i].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_score:.3f}')
                    axes[i].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
                    axes[i].set_xlabel('False Positive Rate')
                    axes[i].set_ylabel('True Positive Rate')
                    axes[i].set_title(f'{class_name}', fontweight='bold')
                    axes[i].legend(loc='lower right')
                    axes[i].grid(True, alpha=0.3)
                else:
                    axes[i].text(0.5, 0.5, 'No positive samples', 
                               ha='center', va='center')
                    axes[i].set_title(f'{class_name}')
            except Exception as e:
                axes[i].text(0.5, 0.5, f'Error: {str(e)}', 
                           ha='center', va='center', fontsize=8)
                axes[i].set_title(f'{class_name}')
    
    # Hide extra subplots if any
    for i in range(len(class_names), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('./output/baseline_roc_curves_grid.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Grid ROC curves saved to: ./output/baseline_roc_curves_grid.png")


# Generate both versions
print("\nGenerating ROC curves...")
print("="*60)

# Combined plot (all curves in one graph)
print("\n1. Combined ROC Curve (all diseases in one graph):")
plot_roc_curves_combined(y_true, predictions, CLASS_NAMES)

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================

results = {
    'model': 'Baseline ResNet-50',
    'methodology': 'El-Fiky et al. (2021)',
    'avg_auc': float(avg_auc),
    'micro_f1': float(micro_f1),
    'macro_f1': float(macro_f1),
    'hamming_loss': float(hamming),
    'per_class_auc': {name: float(auc) for name, auc in zip(CLASS_NAMES, per_class_aucs)},
    'training_epochs': len(history.history['loss']),
    'best_epoch': int(np.argmax(history.history['val_auc'])) + 1,
    'best_val_auc': float(max(history.history['val_auc']))
}

import json
with open('./output/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to: ./output/baseline_results.json")

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("BASELINE RESNET-50 TRAINING COMPLETE")
print("="*80)
print("\nComparison with Paper (El-Fiky et al. 2021):")
print(f"  Average AUC:      {avg_auc:.4f} vs 0.9110 (Paper) [{avg_auc-0.911:+.4f}]")
print(f"  Micro F1-score:   {micro_f1:.4f} vs 0.6600 (Paper) [{micro_f1-0.66:+.4f}]")
print(f"  Hamming Loss:     {hamming:.4f} vs 0.4040 (Paper) [{hamming-0.404:+.4f}]")

if avg_auc >= 0.90:
    print("\n✓ Successfully achieved comparable performance to paper!")
elif avg_auc >= 0.88:
    print("\n~ Close to paper performance - acceptable baseline")
else:
    print("\n⚠ Performance below paper - consider:")
    print("  - Increasing training epochs")
    print("  - Adjusting learning rate schedule")
    print("  - Using more data augmentation")

print("\nModel files saved:")
print("  - ./output/baseline_resnet50_best.keras")
print("  - ./output/baseline_resnet50_final.keras")
print("  - ./output/baseline_results.json")
print("  - ./output/baseline_training_history.png")
print("  - ./output/baseline_roc_curves.png")
print("="*80)